In [0]:
%sql
USE dbjahan.silver_data;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_fel AS
SELECT
  'antal_kommuner' AS kontroll,
  CONCAT(COUNT(*), ' kommuner, förväntat 290') AS utfall
FROM dbjahan.silver_data.kommun_elomrade
HAVING COUNT(*) <> 290

UNION ALL

SELECT
  'heby_1917',
  'gamla Hebykoden 1917 finns kvar'
FROM dbjahan.silver_data.kommun_elomrade
WHERE kommun_kod = '1917'

UNION ALL

SELECT
  'null_elomrade',
  CONCAT(COUNT(*), ' kommuner saknar elområde')
FROM dbjahan.silver_data.kommun_elomrade
WHERE elomrade IS NULL
HAVING COUNT(*) > 0

UNION ALL

SELECT
  'fel_elomrade',
  CONCAT(kommun_namn, ' (', kommun_kod, ') är ', elomrade, ', förväntat ', ratt)
FROM dbjahan.silver_data.kommun_elomrade
JOIN (
  SELECT '2180' AS kommun_kod, 'SE3' AS ratt UNION ALL
  SELECT '2104', 'SE3' UNION ALL
  SELECT '2181', 'SE3' UNION ALL
  SELECT '2183', 'SE2' UNION ALL
  SELECT '0880', 'SE4' UNION ALL
  SELECT '0883', 'SE3' UNION ALL
  SELECT '0180', 'SE3' UNION ALL
  SELECT '1280', 'SE4' UNION ALL
  SELECT '2584', 'SE1'
) v USING (kommun_kod)
WHERE elomrade <> ratt

UNION ALL

SELECT
  'elpris_timmar',
  CONCAT(elomrade, ' ', ar, ' har ', n_timmar, ' timmar, förväntat cirka 8760')
FROM dbjahan.silver_data.elpris
WHERE n_timmar NOT BETWEEN 8700 AND 8900;


In [0]:
fel = spark.table("silver_fel").collect()
if fel:
    text = "; ".join(f"{r.kontroll}: {r.utfall}" for r in fel)
    raise RuntimeError(f"silver_kontroll avbruten. {text}")
print("silver_kontroll ok")

silver_kontroll ok
